# Hawkes processes in the lab

Section 1.3 of the lecture notes, at the keyboard and with the package open.  One two-type
Hawkes process with an exponential kernel, read as the pair of market orders of the six-type
order flow of Definition 1.4.6: type 1 is the market buys and type 2 the market sells, with
pressure $\varpi = (+1, -1)$.  Two types carry every computation that §1.4 makes with six, and
their matrices fit on a line.

We take four parametrisations: one with little excitation, one in which each type mostly excites
itself, one in which each mostly excites the other, and one in which the buys excite the sells
while the sells excite only themselves.  For each statistic that the forecast of §1.4 uses, we
read the function of the package that computes it, compute it again in a plainer way, and set the
two side by side.

Section 1 goes with §1.3.3 and Proposition 1.3.21, section 2 with §1.3.4.1, and sections 3 and 4
with §1.3.4.3.

In [ ]:
import inspect

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.linalg import expm, solve_continuous_lyapunov

from unito26.lob import config
from unito26.lob.hawkes import (
    Clock, ExponentialHawkes, HawkesParams, _mean_response_blocks, decayed_counts_at,
    mean_response, mean_response_integral,
)
from unito26.lob.monte_carlo import (
    Contracted, Fan, ForwardCounts, ForwardState, ForwardStatistic, SampleMean,
)
from unito26.lob.simulate import EventType
from unito26.lob.visualization import (
    INK, MUTED, PALETTE, counting_figure, matrix_figure, monte_carlo_figure, raster_figure,
    state_figure, use_template,
)

use_template()

TYPES = ("market buy", "market sell")
PRESSURE = np.array(
    [EventType.MARKET_BUY.pressure, EventType.MARKET_SELL.pressure], dtype=float
)
SWAP = np.array([[0.0, 1.0], [1.0, 0.0]])
IDENTITY = np.eye(2)


def source(function):
    '''Print the source of a function or of a class of the package.'''
    print(inspect.getsource(function))


def show(vector):
    return "(" + ", ".join(f"{value:.4g}" for value in vector) + ")"

## 1. Exponential-kernel Hawkes processes

A parametrisation is a triple $(\mu, A, \beta)$ of baselines, excitation matrix and decay
(Definition 1.3.9).  The package keeps each one in `config` as a list of records, one per entry
$A_{e,e'}$: `Component` is the type excited, $e$, and `Cause` the type exciting it, $e'$, so that
the row is excited and the column excites, as in Remark 1.3.7.  The baseline sits on the diagonal
records and the decay repeats.  The package counts types from 0, so component 0 is the market
buys.  A loader turns the records into a `HawkesParams`, which can also be built directly.

In [ ]:
print(pd.DataFrame(config.ONE_WAY_PAIR_PARAMS).to_string(index=False))

one_way = config.one_way_pair_params()
by_hand = HawkesParams(baseline=[0.4, 0.1], excitation=[[0.8, 0.0], [4.0, 2.4]], decay=4.0)

print()
print("mu  ", one_way.baseline)
print("A   ", one_way.excitation.tolist())
print("beta", one_way.decay)
print("the loader and the constructor agree:", one_way.to_records() == by_hand.to_records())

The four pairs follow, with what the parameters determine: the branching matrix
$\Gamma = A/\beta$ of (1.3.12) and its spectral radius $\rho$, the column sums of $\Gamma$, which
count the direct offspring of one event of each type, the stationary intensity
$\lambda^* = (I - \Gamma)^{-1}\mu$ of (1.3.18), the stationary mean of the state
$m = \lambda^*/\beta$ of Proposition 1.3.21, and the relaxation time $1/(\beta(1-\rho))$ of
Definition 1.3.36.  The last column says whether the pair is direction-symmetric in the sense of
Definition 1.4.12, $\Sigma A \Sigma = A$ and $\Sigma\mu = \mu$, with $\Sigma$ the exchange of the
two types.

In [ ]:
PAIRS = {
    "weakly exciting": config.weakly_exciting_pair_params(),
    "self-exciting": config.self_exciting_pair_params(),
    "cross-exciting": config.cross_exciting_pair_params(),
    "one-way": config.one_way_pair_params(),
}


def direction_symmetric(params):
    '''Definition 1.4.12: the specification is unchanged by the exchange of the two types.'''
    return bool(
        np.allclose(SWAP @ params.excitation @ SWAP, params.excitation)
        and np.allclose(SWAP @ params.baseline, params.baseline)
    )


print(pd.DataFrame([
    {
        "pair": name,
        "rho": round(params.branching_ratio, 4),
        "column sums of Gamma": show(params.branching_matrix.sum(axis=0)),
        "mu": show(params.baseline),
        "lambda*": show(params.stationary_intensity()),
        "m": show(params.stationary_mean()),
        "relaxation time (s)": round(params.relaxation_time(), 4),
        "direction-symmetric": direction_symmetric(params),
    }
    for name, params in PAIRS.items()
]).to_string(index=False))

The three symmetric pairs have one event a second of each type and differ in how much of it is
excited, and by whom.  The one-way pair has the same total rate, three quarters of it in sells.
Its $\Gamma$ is triangular, so its eigenvalues are its diagonal entries and $\rho = 0.6$ is the
self-excitation of the sells alone: the entry $\Gamma_{2,1} = 1$ does not enter the spectrum, and
it is what makes the first column sum exceed one and $\lambda^*_2$ three times $\lambda^*_1$.

The branching matrices, on one colour scale:

In [ ]:
matrix_figure(
    {name: params.branching_matrix for name, params in PAIRS.items()}, TYPES,
    "Gamma: the expected number of direct offspring, of the row type, of one event of the column type",
)

The excitation matrices are the same picture in another unit, since $A = \beta\Gamma$ with
$\beta = 4$ in all four: an entry of $\Gamma$ counts offspring, and the same entry of $A$ is the
jump of an intensity, in events per second (Remark 1.3.10).

In [ ]:
matrix_figure(
    {name: params.excitation for name, params in PAIRS.items()}, TYPES,
    "A (1/s): the jump in the intensity of the row type at one event of the column type",
)

A simulator holds a parametrisation and a random generator, and `events(horizon)` yields the
pairs (time, type) of a path started from the empty state, up to the horizon.  We draw one path
per pair and leave out the first twenty seconds, long against every relaxation time above.

In [ ]:
WARM_UP = 20.0
STRETCH = (WARM_UP, WARM_UP + 30.0)


def simulate(params, horizon, seed):
    '''One path on (0, horizon], started empty: its event times and types.'''
    events = list(ExponentialHawkes(params, rng=seed).events(horizon))
    times = np.array([time for time, _ in events], dtype=float)
    types = np.array([kind for _, kind in events], dtype=int)
    return times, types


def within(times, types, interval):
    '''The events of a path that fall in the interval (a, b].'''
    kept = (times > interval[0]) & (times <= interval[1])
    return times[kept], types[kept]


SAMPLE_PATHS = {
    name: simulate(params, STRETCH[1], seed)
    for (name, params), seed in zip(PAIRS.items(), (1, 2, 3, 4))
}

figure = raster_figure(
    {name: within(*path, STRETCH) for name, path in SAMPLE_PATHS.items()}, TYPES,
    "Thirty seconds of each pair, events by type",
)
figure.update_xaxes(range=STRETCH)
figure

The four rows have about the same number of events and arrange them differently: evenly in the
first, in runs of one colour in the second, in alternating runs in the third, and in runs of sells
that a buy sets off in the fourth.

The next four figures show where the arrangement comes from.  Each has the intensity
$\lambda(t) = \mu + A Z(t)$ of (1.3.13) on top, the state $Z$ of (1.3.11) in the middle, with a
dotted line at its stationary mean $m$, and the events beneath.  The state jumps by one at an
event of its own type and decays at the rate $\beta$ in between, in every pair: the parametrisation
enters $Z$ only through the times of the events.  It enters the intensity through $A$, which
turns the same state into different rates.

In [ ]:
ZOOM = Clock(np.linspace(WARM_UP, WARM_UP + 15.0, 1501))

state_figure(
    PAIRS["weakly exciting"], *SAMPLE_PATHS["weakly exciting"], ZOOM, TYPES,
    "Weakly exciting: the intensities stay close to the baseline 0.9",
)

With the self-exciting pair an event raises the intensity of its own type by 2.8 and that of the
other by 0.2, so a type that has just fired is the one most likely to fire next.

In [ ]:
state_figure(
    PAIRS["self-exciting"], *SAMPLE_PATHS["self-exciting"], ZOOM, TYPES,
    "Self-exciting: the intensity of a type follows its own state",
)

The cross-exciting pair has the rows of $A$ exchanged: the intensity of the buys now follows the
state of the sells.

In [ ]:
state_figure(
    PAIRS["cross-exciting"], *SAMPLE_PATHS["cross-exciting"], ZOOM, TYPES,
    "Cross-exciting: the intensity of a type follows the state of the other",
)

In the one-way pair a buy raises the intensity of the sells by 4 and its own by 0.8, and a sell
raises only the intensity of the sells, by 2.4.  The intensity of the buys does not move at a
sell.

In [ ]:
state_figure(
    PAIRS["one-way"], *SAMPLE_PATHS["one-way"], ZOOM, TYPES,
    "One-way: the buys excite the sells, and the sells only themselves",
)

The dotted lines are computed by `stationary_mean`, as $\lambda^*/\beta$.  Proposition 1.3.21
also gives the identity $(\beta I - A)\,m = \mu$ of (1.3.20), which is checked on each pair.

In [ ]:
for name, params in PAIRS.items():
    m = params.stationary_mean()
    left = (params.decay * IDENTITY - params.excitation) @ m
    print(f"{name:16s} m = {show(m):18s} (beta I - A) m = {show(left):12s} mu = {show(params.baseline)}")

## 2. Second-order structure

Theorem 1.3.29 gives the stationary covariance of the state, $V$, as the solution of the Lyapunov
equation (1.3.26),

$$K V + V K^\top + \operatorname{diag}(\lambda^*) = 0, \qquad K = A - \beta I .$$

The equation is linear in $V$.  Stacking the rows of $V$ into a vector,
$\operatorname{vec}(KV) = (K \otimes I)\operatorname{vec}(V)$ and
$\operatorname{vec}(VK^\top) = (I \otimes K)\operatorname{vec}(V)$, so (1.3.26) is a system of
four equations in the four entries of $V$:

$$(K \otimes I + I \otimes K)\,\operatorname{vec}(V) = -\operatorname{vec}\big(\operatorname{diag}(\lambda^*)\big).$$

We solve it as such on the one-way pair.

In [ ]:
def covariance_by_linear_system(params):
    '''V from (1.3.26), written as a linear system in the entries of V.'''
    K = params.hurwitz_matrix
    system = np.kron(K, IDENTITY) + np.kron(IDENTITY, K)
    forcing = -np.diag(params.stationary_intensity()).reshape(-1)
    return np.linalg.solve(system, forcing).reshape(2, 2)


one_way = PAIRS["one-way"]
K = one_way.hurwitz_matrix

print("K =", K.tolist())
print()
print("K (x) I + I (x) K =")
print(np.kron(K, IDENTITY) + np.kron(IDENTITY, K))
print()
print("V =", covariance_by_linear_system(one_way).round(6).tolist())

The package does not build that system.  With $d$ types it has $d^2$ unknowns, and solving it
costs of the order of $d^6$ operations.  `scipy.linalg.solve_continuous_lyapunov` works on the
$d \times d$ matrices themselves, by the Bartels–Stewart algorithm, at a cost of the order of
$d^3$.  Its convention is to solve $aX + Xa^{H} = q$, so the forcing term goes to the right-hand
side with a minus sign.

In [ ]:
source(HawkesParams.stationary_covariance)

V = one_way.stationary_covariance()
arrivals = np.diag(one_way.stationary_intensity())

print("V =", V.round(6).tolist())
print("largest difference from the linear system:", np.abs(V - covariance_by_linear_system(one_way)).max())
print("largest residual of (1.3.26):             ", np.abs(K @ V + V @ K.T + arrivals).max())
print()
print("with q = +diag(lambda*):", solve_continuous_lyapunov(K, arrivals).round(6).tolist())
print("with K transposed:      ", solve_continuous_lyapunov(K.T, -arrivals).round(6).tolist())

The last two lines are the two mistakes the call admits.  The wrong sign returns $-V$, and a
negative variance gives it away.  The transposed $K$ returns a symmetric positive-definite
matrix, with the variance of the buys seven times too large and nothing to give it away.  On the
three symmetric pairs $K = K^\top$ and the mistake cannot be seen at all, which is why the
reconciliation is run on the one-way pair.

For the Monte Carlo we reach the stationary law as the proof of the theorem does, from the empty
state.  A `Fan` is a set of independent paths continued from one state, here $Z(0+) = 0$, and
`ForwardState` reads $Z$ on each of them; both belong to the module `monte_carlo`, which section 4
reads.  The state is read at one time, eight relaxation times after the start, where the bias of
the empty start, of the order of $e^{-8}$, is far below the Monte Carlo error.  The mean $m$ is
known, so an entry of $V$ is estimated by the mean of a product of two centred readings, and its
standard error is the one of those products.  The state is heavy-tailed, and a standard error
computed as if it were Gaussian would be several times too small.  The three entries of a pair
are read on the same paths, so their scores move together; and the estimate is skewed low, a fan
without a large excursion underestimating the covariance and its standard error at once.

In [ ]:
COVARIANCE_FAN_SIZE = 20_000

rows = []
for (name, params), seed in zip(PAIRS.items(), (21, 22, 23, 24)):
    late = 8 * params.relaxation_time()
    fan = Fan.sample(params, np.zeros(2), late, COVARIANCE_FAN_SIZE, seed)
    centred = ForwardState().read(fan, Clock(np.array([late])))[:, 0] - params.stationary_mean()
    estimate = SampleMean.from_sample(centred[:, :, None] * centred[:, None, :])
    V = params.stationary_covariance()
    scores = estimate.z_scores(V)
    for i, j in ((0, 0), (0, 1), (1, 1)):
        rows.append({
            "pair": name, "entry": f"V[{i + 1},{j + 1}]", "formula": V[i, j],
            "Monte Carlo": estimate.mean[i, j], "SE": estimate.standard_error[i, j],
            "z": scores[i, j],
        })

print(pd.DataFrame(rows).round(4).to_string(index=False))

## 3. Computing the mean response

By Corollary 1.3.35 the mean response to an event of type $e'$ is
$R_{e'}(h) = e^{Kh}\mathbf e_{e'}$, the $e'$-th column of a matrix exponential.  The exponential
of a square matrix is defined by the series

$$e^{Kh} = \sum_{k \ge 0} \frac{(Kh)^k}{k!},$$

and it is the solution of $X'(h) = K X(h)$ with $X(0) = I$.  We compute it in three ways.

*The series.*  $K = A - \beta I$ and $\beta I$ commutes with $A$, so
$e^{Kh} = e^{-\beta h} e^{Ah}$.  The series of $e^{Ah}$ has non-negative terms, since $A \ge 0$.

*The eigenvalues.*  If $K = P \operatorname{diag}(\kappa) P^{-1}$ then
$e^{Kh} = P \operatorname{diag}(e^{\kappa_i h}) P^{-1}$.  The eigenvalues of $K$ are
$\kappa_i = \beta(\gamma_i - 1)$, with $\gamma_i$ those of $\Gamma$, so every entry of $e^{Kh}$ is
a combination of exponentials decaying at the rates $\beta(1 - \gamma_i)$.

*The library.*  `scipy.linalg.expm`, which scales the matrix down, uses a rational approximation
of the exponential, and squares the result back up.

In [ ]:
def exponential_by_series(params, h, terms):
    '''e^{Kh} as e^{-beta h} sum_k (Ah)^k / k!, with the series cut after `terms` terms.'''
    term, total = IDENTITY, IDENTITY.copy()
    for k in range(1, terms):
        term = term @ (params.excitation * h) / k
        total = total + term
    return np.exp(-params.decay * h) * total


def exponential_by_eigenvalues(params, h):
    '''P diag(e^{kappa h}) P^{-1}, for K = P diag(kappa) P^{-1}.'''
    kappa, P = np.linalg.eig(params.hurwitz_matrix)
    return ((P * np.exp(kappa * h)) @ np.linalg.inv(P)).real


h = 0.5
print("eigenvalues of K:", np.linalg.eigvals(K).real)
print("beta (gamma - 1):", one_way.decay * (np.linalg.eigvals(one_way.branching_matrix).real - 1))
print()
print("series, 30 terms:\n", exponential_by_series(one_way, h, 30))
print("eigenvalues:\n", exponential_by_eigenvalues(one_way, h))
print("scipy.linalg.expm:\n", expm(K * h))

The series was not summed on $K$ itself.  The powers of $Kh$ alternate in sign and grow before
the factorial overtakes them, so the terms cancel, and at a long horizon the rounding of the
large terms is all that is left of the small sum.

In [ ]:
def exponential_by_naive_series(params, h, terms):
    '''sum_k (Kh)^k / k!, summed as it is written.'''
    term, total = IDENTITY, IDENTITY.copy()
    for k in range(1, terms):
        term = term @ (params.hurwitz_matrix * h) / k
        total = total + term
    return total


cross = PAIRS["cross-exciting"]
rows = []
for horizon in (1.0, 5.0, 10.0):
    exact = expm(cross.hurwitz_matrix * horizon)
    rows.append({
        "h (s)": horizon,
        "relative error, series on K": np.abs(exponential_by_naive_series(cross, horizon, 400) - exact).max() / exact.max(),
        "relative error, series on A": np.abs(exponential_by_series(cross, horizon, 400) - exact).max() / exact.max(),
    })
print(pd.DataFrame(rows).to_string(index=False))

The package calls the library.  `mean_response` returns $e^{Kh}$ whole, and its column $e'$ is
$R_{e'}(h)$.

In [ ]:
source(mean_response)

R = one_way.mean_response(h)
print("e^{Kh} at h = 0.5:")
print(R)
print()
print("R_1(h), the response to one more market buy: ", R[:, 0])
print("R_2(h), the response to one more market sell:", R[:, 1])
print()
print("at h = 0:", one_way.mean_response(0.0).tolist())
print("e^{K 0.2} e^{K 0.3} against e^{K 0.5}:", np.abs(one_way.mean_response(0.2) @ one_way.mean_response(0.3) - R).max())
print()
print("expm(K) * h, which is another matrix:")
print(expm(K) * h)

The forward means of §1.3.4.3 also need the integral of the response,

$$\Phi(h) = \int_0^h e^{Ku}\,du ,$$

and the integral of $\Phi$.  Two plain routes to $\Phi$: the trapezoidal rule on a grid, and the
closed form $K^{-1}(e^{Kh} - I)$, which holds when $K$ is invertible, that is when $\rho < 1$
(Proposition 1.3.27).

In [ ]:
def integral_by_quadrature(excitation, decay, h, steps):
    '''The trapezoidal rule for the integral of e^{Ku} over [0, h], on a regular grid.'''
    grid = Clock(np.linspace(0.0, h, steps + 1))
    values = mean_response(excitation, decay, grid)
    return (values[:-1] + values[1:]).sum(axis=0) * (h / steps) / 2


quadrature = integral_by_quadrature(one_way.excitation, one_way.decay, h, 2000)
closed_form = np.linalg.solve(K, expm(K * h) - IDENTITY)

print("Phi(h) by quadrature:\n", quadrature)
print("Phi(h) = K^{-1} (e^{Kh} - I):\n", closed_form)

CRITICAL = 4.0 * np.array([[0.5, 0.5], [0.5, 0.5]])
print()
print("at rho = 1, det K =", np.linalg.det(CRITICAL - 4.0 * IDENTITY))
try:
    np.linalg.solve(CRITICAL - 4.0 * IDENTITY, expm((CRITICAL - 4.0 * IDENTITY) * h) - IDENTITY)
except np.linalg.LinAlgError as error:
    print("the closed form raises:", error)

The package takes a third route, which needs no inverse and gives the three matrices in one call.
Let

$$G = \begin{pmatrix} K & I & 0 \\ 0 & 0 & I \\ 0 & 0 & 0 \end{pmatrix},$$

a matrix of $3d \times 3d$ entries, and let $(E, F, D)$ be the first block row of $e^{Gh}$.
Since $\frac{d}{dh} e^{Gh} = e^{Gh} G$, and the first block row of $e^{Gh}G$ is $(EK, E, F)$,

$$E' = EK, \qquad F' = E, \qquad D' = F,$$

with $(E, F, D) = (I, 0, 0)$ at $h = 0$.  Therefore

$$E = e^{Kh}, \qquad F = \Phi(h), \qquad D = \int_0^h \Phi(u)\,du .$$

In [ ]:
d = 2
G = np.zeros((3 * d, 3 * d))
G[:d, :d] = K
G[:d, d:2 * d] = IDENTITY
G[d:2 * d, 2 * d:] = IDENTITY
print("G =")
print(G)

blocks = expm(G * h)
print()
print("e^{Gh} =")
print(blocks.round(4))

E, F, D = blocks[:d, :d], blocks[:d, d:2 * d], blocks[:d, 2 * d:]
grid = Clock(np.linspace(0.0, h, 2001))
integrals = mean_response_integral(one_way.excitation, one_way.decay, grid)
double = (integrals[:-1] + integrals[1:]).sum(axis=0) * (h / 2000) / 2
print()
print("E against expm(K h):                  ", np.abs(E - expm(K * h)).max())
print("F against the closed form of Phi:     ", np.abs(F - closed_form).max())
print("F against the quadrature of e^{Ku}:   ", np.abs(F - quadrature).max())
print("D against the quadrature of Phi:      ", np.abs(D - double).max())

This is `_mean_response_blocks`.  The public functions `mean_response` and
`mean_response_integral` return its first and second block, and every forward mean of section 4
is built from the three.  The horizon may be a whole clock of horizons: `expm` exponentiates a
stack of matrices in one call, and each block then has a leading axis.  The functions of the
module take $A$ and $\beta$ rather than a `HawkesParams`, which refuses $\rho \ge 1$, and so they
serve at the critical point too.

In [ ]:
source(_mean_response_blocks)
source(mean_response_integral)

clock = Clock(np.linspace(0.0, 3.0, 301))
on_clock = one_way.mean_response_integral(clock)
looped = np.array([one_way.mean_response_integral(horizon) for horizon in clock])
print("Phi on a clock of 301 horizons:", on_clock.shape, " against a loop over them:", np.abs(on_clock - looped).max())
print()
print("Phi(0.5) at rho = 1, from the blocks:")
print(mean_response_integral(CRITICAL, 4.0, h))
print("and by quadrature:")
print(integral_by_quadrature(CRITICAL, 4.0, h, 2000))

## 4. Timescales

§1.3.4.3 asks what the state and the counts are expected to be at a time $t$, given the state
just after an earlier time $s$.  Its formulae depend on $s$ and $t$ only through $t - s$, and the
code of this section has no $s$ in it at all.  The reason is the paragraph that follows
Theorem 1.3.14 in the notes.  The state $Z$ is a Markov process whose rates depend neither on the
time nor on the counts, and $N$ counts its jumps.  So, given the state $z$ just after a time $s$,
the future increments of $N$ and the future of $Z$ have a law that depends on $z$ and on the time
elapsed, and on nothing else: not on $s$, not on $N(s)$, not on the path before $s$.  A statement
about $(s, t]$ given $Z(s+) = z$ is therefore a statement about $(0, t - s]$ for the process
continued from $z$ with the clock moved to zero.

Three things go with it.  The time $s$ may be fixed or a stopping time, and not a time chosen by
looking ahead.  When $s$ is the time of an event, $z$ is the state after its jump,
$Z(s+) = Z(s) + \mathbf e_E$.  And the statement is conditional: averaged over the law of $Z(s)$,
which changes with $s$ unless the process is stationary, it gives an unconditional law that does
depend on $s$.  Nothing is assumed of $\rho$.

Proposition 1.3.33 gives the two means.  For the state, (1.3.30),

$$m_{s,s+h}(z) = e^{Kh} z + \Phi(h)\,\mu ,$$

and for the counts, (1.3.31),

$$\mathbb E\big[N(s+h) - N(s) \mid Z(s+) = z\big]
= h\,\mu + A\,\Phi(h)\,z + A\Big(\int_0^h \Phi(u)\,du\Big)\mu .$$

The package returns each as its terms, from the three blocks of section 3.

In [ ]:
source(HawkesParams.forward_mean_state_terms)
source(HawkesParams.forward_mean_counts_terms)

The three terms of the counts count three disjoint kinds of event.  Given the past, the events
after $s$ are the immigrants, which arrive at the rate $\mu$; the events descended from an event
at or before $s$, whose excitation the state carries; and the events descended from those
immigrants.  The fields are named accordingly: `immigrants`, `descendants_of_state`,
`descendants_of_immigrants`.  For the state the two terms are `from_state`, what is left of the
weight the state carries together with the weight of its descendants, and `from_baseline`, the
weight of the immigrants and of theirs.

### The state

We take the one-way pair and one long path.  The time $s$ is the first event after the warm-up at
which the total state after the jump exceeds a level: a stopping time, decided by the path up to
$s$ alone.  `decayed_counts_at` reads $Z$ before the jump of an event that falls on its clock, so
$z$ is the reading plus the unit vector of the event.

In [ ]:
LEVEL = 3.0
AHEAD = 3.0

times, types = simulate(one_way, 300.0, 41)
after_jump = decayed_counts_at(one_way.decay, times, types, np.zeros(2), Clock(times)) + IDENTITY[types]
reached = (times > WARM_UP) & (after_jump.sum(axis=1) > LEVEL)
assert reached.any(), "the path does not reach the level"
stopped = np.argmax(reached)
s, z = times[stopped], after_jump[stopped]
m = one_way.stationary_mean()

print(f"s = {s:.4f} s, at a {TYPES[types[stopped]]}")
print("z = Z(s+) =", show(z))
print("m         =", show(m))

The figure shows the path around $s$, and from $s$ onwards the forward mean $m_{s,s+h}(z)$ of the
state, dashed, which is the `total` of the two terms.

In [ ]:
HORIZONS = Clock(np.linspace(0.0, AHEAD, 301))

state_terms = one_way.forward_mean_state_terms(z, HORIZONS)

figure = state_figure(
    one_way, times, types, Clock(np.linspace(s - 2.0, s + AHEAD, 1001)), TYPES,
    "One-way pair: the path through s, and the mean state given Z(s+), dashed",
)
for slot, label in enumerate(TYPES):
    figure.add_trace(
        go.Scatter(
            x=s + HORIZONS, y=state_terms.total[:, slot], mode="lines",
            name=f"{label}, forward mean", line=dict(color=PALETTE[slot], width=2.5, dash="dash"),
        ),
        row=2, col=1,
    )
figure.add_vline(x=s, line=dict(color=INK, width=1, dash="dot"))
figure

The path after $s$ is one draw from the law of which the dashed line is the mean, so the two are
not expected to coincide; the mean is set against many draws at the end of the section.  The
dashed lines start at $z$ and end on the dotted ones, at $m$.

The two terms, and the form (1.3.32), $m_{s,s+h}(z) = m + e^{Kh}(z - m)$, which
`forward_mean_state_about_the_mean` returns as `stationary` and `deviation`:

In [ ]:
state_about = one_way.forward_mean_state_about_the_mean(z, HORIZONS)

figure = make_subplots(rows=1, cols=2, subplot_titles=TYPES, horizontal_spacing=0.1)
for slot in range(2):
    column = slot + 1
    for name, values, colour, dash in (
        ("from_state", state_terms.from_state, PALETTE[2], "solid"),
        ("from_baseline", state_terms.from_baseline, PALETTE[3], "solid"),
        ("total", state_terms.total, INK, "dash"),
    ):
        figure.add_trace(
            go.Scatter(
                x=HORIZONS, y=values[:, slot], mode="lines", name=name, legendgroup=name,
                showlegend=slot == 0, line=dict(color=colour, width=2, dash=dash),
            ),
            row=1, col=column,
        )
    figure.add_hline(y=m[slot], line=dict(color=INK, width=1, dash="dot"), row=1, col=column)
    figure.update_xaxes(title_text="h (s)", row=1, col=column)
figure.update_yaxes(title_text="mean state")
figure.update_layout(
    title="The two terms of the mean state against the horizon, and the stationary mean, dotted",
    height=400, legend=dict(orientation="h", y=1.2, x=0),
)
figure

In [ ]:
print("the two forms of the mean state differ by at most:", np.abs(state_about.total - state_terms.total).max())
print("deviation at h = 0:", show(state_about.deviation[0]), " which is z - m =", show(z - m))
print("deviation at h = 3:", show(state_about.deviation[-1]))

### The counts

The same path counted from $s$, with the expected counts of (1.3.31) dashed.

In [ ]:
count_terms = one_way.forward_mean_counts_terms(z, HORIZONS)

figure = counting_figure(
    {"N(s + h) - N(s)": (times, types)}, (s, s + AHEAD), TYPES,
    "One-way pair: the counts after s, and their mean given Z(s+), dashed",
)
for slot, label in enumerate(TYPES):
    figure.add_trace(
        go.Scatter(
            x=s + HORIZONS, y=count_terms.total[:, slot], mode="lines",
            name=f"{label}, forward mean", line=dict(color=PALETTE[slot], width=2.5, dash="dash"),
        ),
        row=1, col=1,
    )
figure.update_layout(height=420)
figure

The three terms, stacked, for each type.

In [ ]:
TERMS = ("immigrants", "descendants_of_state", "descendants_of_immigrants")

figure = make_subplots(rows=1, cols=2, subplot_titles=TYPES, horizontal_spacing=0.1)
for slot in range(2):
    for name, colour in zip(TERMS, (PALETTE[3], PALETTE[2], PALETTE[4])):
        figure.add_trace(
            go.Scatter(
                x=HORIZONS, y=getattr(count_terms, name)[:, slot], mode="lines", name=name,
                legendgroup=name, showlegend=slot == 0, stackgroup=str(slot),
                line=dict(color=colour, width=1),
            ),
            row=1, col=slot + 1,
        )
    figure.update_xaxes(title_text="h (s)", row=1, col=slot + 1)
figure.update_yaxes(title_text="expected events in (s, s + h]")
figure.update_layout(
    title="The three terms of the expected counts, stacked",
    height=420, legend=dict(orientation="h", y=1.2, x=0),
)
figure

In [ ]:
rows = []
for horizon in (0.1, 1.0, 3.0, 30.0):
    terms = one_way.forward_mean_counts_terms(z, horizon)
    for slot, label in enumerate(TYPES):
        rows.append({
            "h (s)": horizon, "type": label, "expected events": terms.total[slot],
            **{name: getattr(terms, name)[slot] / terms.total[slot] for name in TERMS},
        })
print(pd.DataFrame(rows).round(3).to_string(index=False))

The last three columns are shares of the total.  For the sells, at a tenth of a second nearly
every expected event descends from the state.  That term is bounded in $h$: it ends at
$\Gamma(I - \Gamma)^{-1} z$, the whole expected progeny of what the state carries.  The other two
grow without bound, and at thirty seconds they are most of the count.  The immigrants stay a
small share of the sells at every horizon, their baseline being 0.1 against a stationary
intensity of 1.5.  For the buys the immigrants are half of the count from the start and four
fifths of it in the end: nothing excites a buy but a buy, and with $\Gamma_{1,1} = 0.2$.

The form (1.3.33) regroups the same expectation as $h\,\lambda^* + A\,\Phi(h)(z - m)$: the count
that the stationary intensity gives, and a deviation that the state carries, bounded in $h$.
`forward_mean_counts_about_the_mean` returns the two.

In [ ]:
count_about = one_way.forward_mean_counts_about_the_mean(z, HORIZONS)
gamma = one_way.branching_matrix
progeny = gamma @ np.linalg.inv(IDENTITY - gamma)

print("the two forms of the expected counts differ by at most:", np.abs(count_about.total - count_terms.total).max())
print("stationary count at h = 3:", show(count_about.stationary[-1]), " which is 3 lambda* =", show(3 * one_way.stationary_intensity()))
print("deviation at h = 3:       ", show(count_about.deviation[-1]))
print("its limit, Gamma (I - Gamma)^-1 (z - m):", show(progeny @ (z - m)))

### Contracted with the pressure

The quantity that §1.4 forecasts is the signed count $\varpi^\top\big(N(s+h) - N(s)\big)$, buys
minus sells, which the order flow imbalance is per unit of mean size (Proposition 1.4.9).  Its
expectation is the contraction of the terms with $\varpi$, and `contracted` returns it term by
term.

On the one-way pair all three terms remain.

In [ ]:
signed = count_terms.contracted(PRESSURE)
signed_about = count_about.contracted(PRESSURE)

figure = go.Figure()
for name, colour in zip(TERMS, (PALETTE[3], PALETTE[2], PALETTE[4])):
    figure.add_scatter(x=HORIZONS, y=getattr(signed, name), mode="lines", name=name,
                       line=dict(color=colour, width=2))
figure.add_scatter(x=HORIZONS, y=signed.total, mode="lines", name="total",
                   line=dict(color=INK, width=2, dash="dash"))
figure.add_scatter(x=HORIZONS, y=signed_about.stationary, mode="lines", name="h varpi' lambda*",
                   line=dict(color=MUTED, width=1.5, dash="dot"))
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="expected buys minus sells in (s, s + h]")
figure.update_layout(
    title="One-way pair: the three terms of the expected signed count",
    height=420, legend=dict(orientation="h", y=1.12, x=0),
)
figure

The immigrants contribute $h\,\varpi^\top\mu = 0.3\,h$, more buys than sells.  The descendants of
the state contribute a bounded term, and the descendants of the immigrants a term whose slope
tends to $-1.3$.  Regrouped by (1.3.33) they are $-h$, the drift $h\,\varpi^\top\lambda^*$ that
the parameters carry whatever the state, plus $\varpi^\top A\,\Phi(h)(z - m)$.

Under direction symmetry the first and third terms are matrices that the exchange leaves
invariant, applied to $\mu$, which it fixes, and $\varpi$ annihilates them (Proposition 1.4.13).
Only the descendants of the state remain, and that term is $\theta(h)\,(z_1 - z_2)$, with
$\theta(h)$ the signed excitation at the horizon of Corollary 1.4.21, which the package computes
as `signed_excitation_at_horizon`.  We evaluate the three symmetric pairs from
the same state $z$.

In [ ]:
SYMMETRIC = {name: params for name, params in PAIRS.items() if direction_symmetric(params)}

rows = []
for name, params in SYMMETRIC.items():
    for horizon in (0.1, 1.0, 3.0):
        terms = params.forward_mean_counts_terms(z, horizon).contracted(PRESSURE)
        theta = params.signed_excitation_at_horizon(PRESSURE, horizon)[0]
        rows.append({
            "pair": name, "h (s)": horizon,
            **{label: getattr(terms, label) for label in TERMS},
            "theta(h) (z1 - z2)": theta * (z[0] - z[1]),
        })
print("z1 - z2 =", round(z[0] - z[1], 4))
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
figure = go.Figure()
for (name, params), colour in zip(SYMMETRIC.items(), (MUTED, PALETTE[0], PALETTE[1])):
    forecast = params.forward_mean_counts_terms(z, HORIZONS).contracted(PRESSURE)
    figure.add_scatter(x=HORIZONS, y=forecast.descendants_of_state, mode="lines", name=name,
                       line=dict(color=colour, width=2))
figure.add_hline(y=0.0, line=dict(color=INK, width=1, dash="dot"))
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="expected buys minus sells in (s, s + h]")
figure.update_layout(
    title="Direction-symmetric pairs: the expected signed count from the same state",
    height=420, legend=dict(orientation="h", y=1.12, x=0),
)
figure

The state has more sells than buys in it.  The self-exciting pair expects more sells to follow,
the cross-exciting pair expects buys, and the weakly exciting pair expects almost nothing either
way.  The sign of the forecast belongs to the kernel, through the sign of $\theta$, and the state
supplies only $z_1 - z_2$.  This is Proposition 1.4.16 on two types; on six the code is the same,
with the pressure vector of the six.

### The mean response

The mean response $R_{e'}(h)$ is the displacement of the mean state by one more event of type
$e'$ in the state.  In the terms above, `from_baseline` does not depend on $z$ and `from_state` is
linear in it, so the displacement is `from_state` at the unit vector, $e^{Kh}\mathbf e_{e'}$,
whatever $z$.

In [ ]:
one_more_buy = one_way.forward_mean_state(z + IDENTITY[0], HORIZONS) - one_way.forward_mean_state(z, HORIZONS)
print("displacement by one more buy, against the first column of e^{Kh}:",
      np.abs(one_more_buy - one_way.mean_response(HORIZONS)[:, :, 0]).max())

figure = make_subplots(rows=1, cols=4, subplot_titles=list(PAIRS), shared_yaxes=True,
                       horizontal_spacing=0.03)
for column, (name, params) in enumerate(PAIRS.items(), start=1):
    response = params.mean_response(HORIZONS)
    for cause, dash in ((0, "solid"), (1, "dash")):
        for slot in range(2):
            label = f"state of the {TYPES[slot]}s after one more {TYPES[cause]}"
            figure.add_trace(
                go.Scatter(
                    x=HORIZONS, y=response[:, slot, cause], mode="lines", name=label,
                    legendgroup=label, showlegend=column == 1,
                    line=dict(color=PALETTE[slot], width=2, dash=dash),
                ),
                row=1, col=column,
            )
    reference = np.exp(-HORIZONS / params.relaxation_time())
    figure.add_trace(
        go.Scatter(
            x=HORIZONS, y=reference, mode="lines", name="exp(-beta (1 - rho) h)",
            legendgroup="reference", showlegend=column == 1,
            line=dict(color=INK, width=1, dash="dot"),
        ),
        row=1, col=column,
    )
    figure.update_xaxes(title_text="h (s)", row=1, col=column)
figure.update_yaxes(title_text="displacement of the mean state", row=1, col=1)
figure.update_layout(
    title="The mean response, entry by entry, and the slowest mode",
    height=460, legend=dict(orientation="h", y=1.45, x=0),
)
figure

The dotted curve decays at the rate $\beta(1-\rho)$ of the relaxation time (Definition 1.3.36).
It is the slowest mode of the response: no entry decays more slowly, and an entry decays at that
rate only where the mode is present (Remark 1.3.37).  In the weakly exciting pair an event is
forgotten in a quarter of a second, at close to the rate $\beta$ of the kernel itself.  In the
self- and cross-exciting pairs the same relaxation time of one second is reached in opposite
ways: the extra event stays in its own type, or passes to the other.  In the one-way pair, from
a fifth of a second on, a buy displaces the state of the sells by more than its own; the state of
the sells after a sell is the dotted curve exactly; and the state of the buys after a buy decays
at twice that rate, the sells never feeding back.

### The formulae against Monte Carlo

The module `monte_carlo` holds the estimates.  A `Fan` is the sampling: independent paths
continued from one state by `ExponentialHawkes.from_state`, with the clock at zero.  A
`ForwardStatistic` is what is read on each path, together with the expectation its mean
estimates; the base class derives the estimate and its scores from those two methods, and a new
statistic is a subclass that states them.

In [ ]:
source(ForwardStatistic)
source(ForwardCounts)

`ForwardState` reads $Z(s+h)$, `ForwardCounts` reads $N(s+h) - N(s)$, and `Contracted` wraps a
statistic and contracts it with a vector path by path, so that the standard error is that of the
signed count.  We draw one fan per pair from the state $z$ of the path above, and read the three
statistics on a clock of horizons that starts after zero: at $h = 0$ every path reads the same
and a standard error of zero scores nothing.

In [ ]:
FAN_SIZE = 5000
VALIDATION = Clock(np.linspace(0.05, AHEAD, 60))
STATISTICS = {
    "state": ForwardState(),
    "counts": ForwardCounts(),
    "signed count": Contracted(ForwardCounts(), PRESSURE),
}

FANS = {
    name: Fan.sample(params, z, AHEAD, FAN_SIZE, seed)
    for (name, params), seed in zip(PAIRS.items(), (51, 52, 53, 54))
}
READINGS = {
    (name, label): statistic.read(fan, VALIDATION)
    for name, fan in FANS.items() for label, statistic in STATISTICS.items()
}
print({name: fan.size for name, fan in FANS.items()})

In [ ]:
figure = monte_carlo_figure(
    VALIDATION, READINGS["one-way", "state"],
    STATISTICS["state"].expectation(FANS["one-way"], VALIDATION), TYPES,
    "One-way pair, mean state from z: fan against (1.3.30)",
)
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="mean of Z(s + h)")
figure

In [ ]:
figure = monte_carlo_figure(
    VALIDATION, READINGS["one-way", "counts"],
    STATISTICS["counts"].expectation(FANS["one-way"], VALIDATION), TYPES,
    "One-way pair, expected counts from z: fan against (1.3.31)",
)
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="mean of N(s + h) - N(s)")
figure

In [ ]:
figure = monte_carlo_figure(
    VALIDATION, READINGS["one-way", "signed count"][:, :, None],
    STATISTICS["signed count"].expectation(FANS["one-way"], VALIDATION)[:, None],
    ["buys minus sells"],
    "One-way pair, expected signed count from z: fan against the contracted terms",
)
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="mean of varpi' (N(s + h) - N(s))")
figure

For all four pairs, the largest score of each estimate against its formula along the clock.  The
bands of the figures are pointwise, and the readings at sixty horizons come from the same paths,
so the scores rise and fall together and the largest of sixty is larger than one score: with a
correct formula it is about 2 at the median, and beyond 4 no more than once in a hundred fans.

In [ ]:
rows = []
for (name, label), reading in READINGS.items():
    formula = STATISTICS[label].expectation(FANS[name], VALIDATION)
    scores = np.abs(SampleMean.from_sample(reading).z_scores(formula))
    rows.append({"pair": name, "statistic": label, "largest |z|": scores.max()})
print(pd.DataFrame(rows).pivot(index="pair", columns="statistic", values="largest |z|")
      .loc[list(PAIRS), list(STATISTICS)].round(3).to_string())

The mean response is a difference of two forward means, and is estimated by two fans of the same
size, one from $z + \mathbf e_{e'}$ and one from $z$.  The fans are independent, so path $k$ of
one may be paired with path $k$ of the other, and the standard error of the mean of the
differences is the one of the difference of the means.

In [ ]:
base = READINGS["one-way", "state"]
differences = {}
for cause, seed in ((0, 61), (1, 62)):
    more = Fan.sample(one_way, z + IDENTITY[cause], AHEAD, FAN_SIZE, seed)
    differences[cause] = ForwardState().read(more, VALIDATION) - base
    scores = SampleMean.from_sample(differences[cause]).z_scores(one_way.mean_response(VALIDATION)[:, :, cause])
    print(f"one more {TYPES[cause]}: largest |z| along the clock {np.abs(scores).max():.3f}")

figure = monte_carlo_figure(
    VALIDATION, differences[0], one_way.mean_response(VALIDATION)[:, :, 0], TYPES,
    "One-way pair, the response to one more market buy: two fans against (1.3.34)",
)
figure.update_xaxes(title_text="h (s)")
figure.update_yaxes(title_text="displacement of the mean state")
figure